# Predicting the 2026 World Cup
Objective: predict not just the winner but scoreline of each individual game at the WC.

## Step 1: data cleaning

In [28]:
import numpy as np
import pandas as pd
import seaborn as sns

df = pd.read_csv('results.csv')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


### Cleaning results

In [29]:
from datetime import date, datetime

# clean data
df['date'] = pd.to_datetime(df['date'], format="%Y-%m-%d")

try:
    df.drop(columns=['city', 'country'], inplace=True)
except:
    print("No city found")
df['home_score'] = pd.to_numeric(df['home_score'], errors='coerce').astype('Int64')
df['away_score'] = pd.to_numeric(df['away_score'], errors='coerce').astype('Int64')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,False
1,1873-03-08,England,Scotland,4,2,Friendly,False
2,1874-03-07,Scotland,England,2,1,Friendly,False
3,1875-03-06,England,Scotland,2,2,Friendly,False
4,1876-03-04,Scotland,England,3,0,Friendly,False


### Calculating ELO

In [30]:
# --- ELO helpers ---------------------------------------------------------

def k_factor(tournament: str) -> float:
    """ELO weight by match importance (modeled on eloratings.net)."""
    t = tournament.lower()
    if 'world cup' in t and 'qualif' not in t:
        return 60.0
    if t in {'copa america', 'uefa euro', 'african cup of nations', 'afc asian cup'}:
        return 50.0
    if 'qualif' in t or 'nations league' in t:
        return 40.0
    if t == 'friendly':
        return 20.0
    return 30.0


def g_multiplier(goal_diff: int) -> float:
    """Goal-difference multiplier — bigger wins move ELO more."""
    gd = abs(goal_diff)
    if gd <= 1:
        return 1.0
    if gd == 2:
        return 1.5
    return (11 + gd) / 8.0


def expected_score(r_home: float, r_away: float, home_adv: float = 100.0) -> float:
    """Probability the home team takes the points (draw counts as 0.5)."""
    return 1.0 / (1.0 + 10.0 ** (-(r_home - r_away + home_adv) / 400.0))


def update_match(r_home, r_away, home_score, away_score, tournament, neutral):
    """Apply one match. Returns the new (home_rating, away_rating)."""
    h = 0.0 if neutral else 100.0
    exp_home = expected_score(r_home, r_away, h)

    if home_score > away_score:
        w_home = 1.0
    elif home_score < away_score:
        w_home = 0.0
    else:
        w_home = 0.5

    k = k_factor(tournament) * g_multiplier(home_score - away_score)
    delta = k * (w_home - exp_home)
    return r_home + delta, r_away - delta


def compute_elo_history(matches: pd.DataFrame, initial_rating: float = 1500.0):
    """Walk matches in date order; stamp each row with pre-match ELO for both teams.

    Returns (df_with_elo, final_ratings_dict). Run over the FULL history
    (not a recent slice) so ratings have time to separate teams.
    """
    matches = matches.sort_values('date', kind='mergesort').reset_index(drop=True)
    ratings: dict[str, float] = {}
    elo_home_pre = np.empty(len(matches))
    elo_away_pre = np.empty(len(matches))

    for i, row in enumerate(matches.itertuples(index=False)):
        home, away = row.home_team, row.away_team
        r_home = ratings.get(home, initial_rating)
        r_away = ratings.get(away, initial_rating)
        elo_home_pre[i] = r_home
        elo_away_pre[i] = r_away

        if pd.isna(row.home_score) or pd.isna(row.away_score):
            continue

        ratings[home], ratings[away] = update_match(
            r_home, r_away,
            int(row.home_score), int(row.away_score),
            row.tournament, bool(row.neutral),
        )

    matches = matches.copy()
    matches['elo_home_pre'] = elo_home_pre
    matches['elo_away_pre'] = elo_away_pre
    return matches, ratings


# --- run over full history, then slice to modeling window ----------------

df, final_ratings = compute_elo_history(df)
# adjust before or after elo, that is the question
df = df[df['date'] > pd.Timestamp("2020-12-31")]

# sanity check: top teams should look roughly sensible
pd.Series(final_ratings).sort_values(ascending=False).head(15)


Spain          2212.508845
Argentina      2168.814153
France         2118.570970
England        2082.359234
Brazil         2063.005074
Colombia       2042.200938
Portugal       2037.689031
Ecuador        2017.082540
Netherlands    2000.655570
Germany        2000.202876
Japan          1990.407273
Morocco        1983.590878
Uruguay        1968.084275
Norway         1967.563905
Turkey         1965.366679
dtype: float64

In [31]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,elo_home_pre,elo_away_pre
43722,2021-01-12,United Arab Emirates,Iraq,0,0,Friendly,False,1560.619013,1735.725996
43723,2021-01-18,Kuwait,Palestine,0,1,Friendly,False,1515.841244,1507.549035
43724,2021-01-19,Dominican Republic,Puerto Rico,0,1,Friendly,False,1387.026266,1195.808819
43725,2021-01-22,Guatemala,Puerto Rico,1,0,Friendly,False,1618.584779,1212.657317
43726,2021-01-25,Dominican Republic,Serbia,0,0,Friendly,False,1370.177768,1828.984250


### Calculating recent Goals for and goals against

In [50]:
from collections import deque

def compute_recent_goals(matches: pd.DataFrame):
    """Walk matches in date order; stamp each row with last-10-games
    goals-for and goals-against, NORMALISED against opponent ELO.

    A team that scores 3 a game against minnows looks identical to one
    that scores 3 a game against top-10 sides if you just use raw totals.
    To fix that, we fit a league-wide baseline `expected_goals_scored =
    a + b*opp_elo` and store (actual - expected) per game. The recent
    stat is then the mean residual over the last 10 games:
        > 0 → out-scoring what an avg team would manage vs those opponents
        < 0 → underperforming for the matchups faced
    Same logic on the GA side; by symmetry the same baseline covers both
    (goals conceded by team = goals scored by opp, and opp's `opp_elo`
    from their POV is this team's elo).
    """
    recent = 10
    matches = matches.sort_values('date', kind='mergesort').reset_index(drop=True)

    # Fit baseline on completed matches; each match yields two observations
    # (one per side), so the fit sees a symmetric (opp_elo, scored) sample.
    played = matches.dropna(subset=['home_score', 'away_score'])
    opp_elo = np.concatenate([played['elo_away_pre'].values,
                              played['elo_home_pre'].values])
    scored = np.concatenate([played['home_score'].astype(float).values,
                             played['away_score'].astype(float).values])
    b, a = np.polyfit(opp_elo, scored, 1)
    print(f"baseline: expected_goals_scored = {a:.3f} + {b:.6f} * opp_elo")

    # Vectorised per-row expectations — pull from these inside the loop.
    exp_home_scored = a + b * matches['elo_away_pre'].values  # home's expected GF
    exp_away_scored = a + b * matches['elo_home_pre'].values  # away's expected GF
    # By construction: home's expected GA == exp_away_scored, and vice versa.

    history: dict[str, deque] = {}
    gf_home_recent = np.empty(len(matches))
    ga_home_recent = np.empty(len(matches))
    gf_away_recent = np.empty(len(matches))
    ga_away_recent = np.empty(len(matches))

    for i, row in enumerate(matches.itertuples(index=False)):
        home, away = row.home_team, row.away_team
        dq_home = history.setdefault(home, deque(maxlen=recent))
        dq_away = history.setdefault(away, deque(maxlen=recent))

        gf_home_recent[i], ga_home_recent[i] = mean_resid(dq_home)
        gf_away_recent[i], ga_away_recent[i] = mean_resid(dq_away)

        if pd.isna(row.home_score) or pd.isna(row.away_score):
            continue

        dq_home.append((
            float(row.home_score) - exp_home_scored[i],
            float(row.away_score) - exp_away_scored[i],
        ))
        dq_away.append((
            float(row.away_score) - exp_away_scored[i],
            float(row.home_score) - exp_home_scored[i],
        ))

    matches = matches.copy()
    matches['gf_home_recent'] = gf_home_recent
    matches['ga_home_recent'] = ga_home_recent
    matches['gf_away_recent'] = gf_away_recent
    matches['ga_away_recent'] = ga_away_recent
    return matches


def mean_resid(dq):
    """Mean (gf_residual, ga_residual) over a team's last-N games. Empty → 0."""
    if not dq:
        return 0.0, 0.0
    n = len(dq)
    gf = sum(g for g, _ in dq) / n
    ga = sum(c for _, c in dq) / n
    return gf, ga


df = compute_recent_goals(df)
df[['date', 'home_team', 'away_team', 'home_score', 'away_score',
    'gf_home_recent', 'ga_home_recent', 'gf_away_recent', 'ga_away_recent']].tail(10)


baseline: expected_goals_scored = 3.811 + -0.001543 * opp_elo


,date,home_team,away_team,home_score,away_score,gf_home_recent,ga_home_recent,gf_away_recent,ga_away_recent
5719,2026-06-26,Cape Verde,Saudi Arabia,<NA>,<NA>,0.252664,-0.048834,0.036565,0.324339
5720,2026-06-26,Uruguay,Spain,<NA>,<NA>,0.013291,0.140835,1.625374,0.129116
5721,2026-06-26,Norway,France,<NA>,<NA>,1.900520,-0.128125,1.314049,0.253206
5722,2026-06-26,Senegal,Iraq,<NA>,<NA>,0.681694,0.057539,-0.112211,-0.344385
5723,2026-06-27,Algeria,Austria,<NA>,<NA>,0.753768,-0.554679,1.161270,-0.495092
5724,2026-06-27,Jordan,Argentina,<NA>,<NA>,0.802347,0.505929,1.112147,-0.155723
5725,2026-06-27,Colombia,Portugal,<NA>,<NA>,1.535042,0.432210,1.489200,0.364084
5726,2026-06-27,DR Congo,Uzbekistan,<NA>,<NA>,-0.145549,-0.823895,0.134871,-0.291430
5727,2026-06-27,Panama,England,<NA>,<NA>,0.600608,0.631491,1.048580,-0.106934
5728,2026-06-27,Croatia,Ghana,<NA>,<NA>,0.945460,0.243921,-0.047854,0.083491


### Calculating recent wins/draws/loses

In [51]:
def compute_recent_wdl(matches: pd.DataFrame):
    recent_games = 5
    matches = matches.sort_values('date', kind='mergesort').reset_index(drop=True)
    recent_wdl: dict[str, deque(maxlen=recent_games)] = {}

    wdl_home_recent = np.empty(len(matches))
    wdl_away_recent = np.empty(len(matches))

    for i, row in enumerate(matches.itertuples(index=False)):
        home, away = row.home_team, row.away_team
        wdl_home = recent_wdl.setdefault(home, deque(maxlen=recent_games))
        wdl_away = recent_wdl.setdefault(away, deque(maxlen=recent_games))

        wdl_home_recent[i] = calc_wdl(wdl_home)
        wdl_away_recent[i] = calc_wdl(wdl_away)

        if pd.isna(row.home_score) or pd.isna(row.away_score):
            continue

        wdl_home.append((row.home_score, row.away_score))
        wdl_away.append((row.away_score, row.home_score))

    matches = matches.copy()
    matches['wdl_home_recent'] = wdl_home_recent
    matches['wdl_away_recent'] = wdl_away_recent
    return matches

def calc_wdl(wdl_team):
    wdl = 0
    for match in wdl_team:
        if match[0] > match[1]:
            wdl += 3
        elif match[0] == match[1]:
            wdl += 1
    return wdl

df = compute_recent_wdl(df)
df.tail(10)


,date,home_team,away_team,home_score,away_score,tournament,neutral,elo_home_pre,elo_away_pre,gf_home_recent,ga_home_recent,gf_away_recent,ga_away_recent,wdl_home_recent,wdl_away_recent,rest_home,rest_away,home_in_host_continent,away_in_host_continent
5719,2026-06-26,Cape Verde,Saudi Arabia,<NA>,<NA>,FIFA World Cup,True,1652.554072,1690.487172,0.252664,-0.048834,0.036565,0.324339,5.0,3.0,5.0,5.0,0,0
5720,2026-06-26,Uruguay,Spain,<NA>,<NA>,FIFA World Cup,True,1968.084275,2212.508845,0.013291,0.140835,1.625374,0.129116,6.0,9.0,5.0,5.0,0,0
5721,2026-06-26,Norway,France,<NA>,<NA>,FIFA World Cup,True,1967.563905,2118.570970,1.900520,-0.128125,1.314049,0.253206,10.0,12.0,4.0,4.0,0,0
5722,2026-06-26,Senegal,Iraq,<NA>,<NA>,FIFA World Cup,True,1918.182608,1749.418812,0.681694,0.057539,-0.112211,-0.344385,9.0,7.0,4.0,4.0,0,0
5723,2026-06-27,Algeria,Austria,<NA>,<NA>,FIFA World Cup,True,1871.678618,1886.563873,0.753768,-0.554679,1.161270,-0.495092,10.0,13.0,5.0,5.0,0,0
5724,2026-06-27,Jordan,Argentina,<NA>,<NA>,FIFA World Cup,True,1779.060631,2168.814153,0.802347,0.505929,1.112147,-0.155723,5.0,15.0,5.0,5.0,0,0
5725,2026-06-27,Colombia,Portugal,<NA>,<NA>,FIFA World Cup,True,2042.200938,2037.689031,1.535042,0.432210,1.489200,0.364084,9.0,10.0,4.0,4.0,0,0
5726,2026-06-27,DR Congo,Uzbekistan,<NA>,<NA>,FIFA World Cup,True,1772.992934,1823.438546,-0.145549,-0.823895,0.134871,-0.291430,10.0,6.0,4.0,4.0,0,0
5727,2026-06-27,Panama,England,<NA>,<NA>,FIFA World Cup,True,1836.620514,2082.359234,0.600608,0.631491,1.048580,-0.106934,8.0,10.0,4.0,4.0,1,0
5728,2026-06-27,Croatia,Ghana,<NA>,<NA>,FIFA World Cup,True,1961.718591,1624.448322,0.945460,0.243921,-0.047854,0.083491,9.0,1.0,4.0,4.0,0,0


### Calculate rest days before game

In [52]:
def compute_rest_days(matches: pd.DataFrame):
    """Walk matches in date order; stamp each row with days since each team's
    previous international match. NaN for a team's first appearance."""
    matches = matches.sort_values('date', kind='mergesort').reset_index(drop=True)
    last_match: dict[str, pd.Timestamp] = {}
    rest_home = np.full(len(matches), np.nan)
    rest_away = np.full(len(matches), np.nan)

    for i, row in enumerate(matches.itertuples(index=False)):
        home, away = row.home_team, row.away_team
        if home in last_match:
            rest_home[i] = (row.date - last_match[home]).days
        if away in last_match:
            rest_away[i] = (row.date - last_match[away]).days

        # stamp first, then update — feature is pre-match
        last_match[home] = row.date
        last_match[away] = row.date

    matches = matches.copy()
    matches['rest_home'] = rest_home
    matches['rest_away'] = rest_away
    return matches


df = compute_rest_days(df)
df[['date', 'home_team', 'away_team', 'rest_home', 'rest_away']].tail(10)


,date,home_team,away_team,rest_home,rest_away
5719,2026-06-26,Cape Verde,Saudi Arabia,5.0,5.0
5720,2026-06-26,Uruguay,Spain,5.0,5.0
5721,2026-06-26,Norway,France,4.0,4.0
5722,2026-06-26,Senegal,Iraq,4.0,4.0
5723,2026-06-27,Algeria,Austria,5.0,5.0
5724,2026-06-27,Jordan,Argentina,5.0,5.0
5725,2026-06-27,Colombia,Portugal,4.0,4.0
5726,2026-06-27,DR Congo,Uzbekistan,4.0,4.0
5727,2026-06-27,Panama,England,4.0,4.0
5728,2026-06-27,Croatia,Ghana,4.0,4.0


### Calculate Host Continent Advantage

In [53]:
# Host-continent advantage: flag whether each team is from the same
# confederation as the country hosting the match. The `country` column
# was dropped earlier, so re-merge it from the raw CSV just for this.

_country_df = pd.read_csv('results.csv', usecols=['date', 'home_team', 'away_team', 'country'])
_country_df['date'] = pd.to_datetime(_country_df['date'], format='%Y-%m-%d')
df = df.merge(_country_df, on=['date', 'home_team', 'away_team'], how='left')

CONTINENT = {
    # UEFA
    'Albania': 'EU', 'Andorra': 'EU', 'Armenia': 'EU', 'Austria': 'EU', 'Azerbaijan': 'EU',
    'Belarus': 'EU', 'Belgium': 'EU', 'Bosnia and Herzegovina': 'EU', 'Bulgaria': 'EU',
    'Croatia': 'EU', 'Cyprus': 'EU', 'Czech Republic': 'EU', 'Czechia': 'EU', 'Denmark': 'EU',
    'England': 'EU', 'Estonia': 'EU', 'Faroe Islands': 'EU', 'Finland': 'EU', 'France': 'EU',
    'Georgia': 'EU', 'Germany': 'EU', 'Gibraltar': 'EU', 'Greece': 'EU', 'Hungary': 'EU',
    'Iceland': 'EU', 'Israel': 'EU', 'Italy': 'EU', 'Kazakhstan': 'EU', 'Kosovo': 'EU',
    'Latvia': 'EU', 'Liechtenstein': 'EU', 'Lithuania': 'EU', 'Luxembourg': 'EU', 'Malta': 'EU',
    'Moldova': 'EU', 'Montenegro': 'EU', 'Netherlands': 'EU', 'North Macedonia': 'EU',
    'Northern Ireland': 'EU', 'Norway': 'EU', 'Poland': 'EU', 'Portugal': 'EU',
    'Republic of Ireland': 'EU', 'Romania': 'EU', 'Russia': 'EU', 'San Marino': 'EU',
    'Scotland': 'EU', 'Serbia': 'EU', 'Slovakia': 'EU', 'Slovenia': 'EU', 'Spain': 'EU',
    'Sweden': 'EU', 'Switzerland': 'EU', 'Turkey': 'EU', 'Ukraine': 'EU', 'Wales': 'EU',
    # CONMEBOL
    'Argentina': 'SA', 'Bolivia': 'SA', 'Brazil': 'SA', 'Chile': 'SA', 'Colombia': 'SA',
    'Ecuador': 'SA', 'Paraguay': 'SA', 'Peru': 'SA', 'Uruguay': 'SA', 'Venezuela': 'SA',
    # CONCACAF
    'Antigua and Barbuda': 'NA', 'Aruba': 'NA', 'Bahamas': 'NA', 'Barbados': 'NA',
    'Belize': 'NA', 'Bermuda': 'NA', 'British Virgin Islands': 'NA', 'Canada': 'NA',
    'Cayman Islands': 'NA', 'Costa Rica': 'NA', 'Cuba': 'NA', 'Curaçao': 'NA', 'Dominica': 'NA',
    'Dominican Republic': 'NA', 'El Salvador': 'NA', 'Grenada': 'NA', 'Guatemala': 'NA',
    'Guyana': 'NA', 'Haiti': 'NA', 'Honduras': 'NA', 'Jamaica': 'NA', 'Martinique': 'NA',
    'Mexico': 'NA', 'Montserrat': 'NA', 'Nicaragua': 'NA', 'Panama': 'NA', 'Puerto Rico': 'NA',
    'Saint Kitts and Nevis': 'NA', 'Saint Lucia': 'NA',
    'Saint Vincent and the Grenadines': 'NA', 'Sint Maarten': 'NA', 'Suriname': 'NA',
    'Trinidad and Tobago': 'NA', 'Turks and Caicos Islands': 'NA', 'United States': 'NA',
    'US Virgin Islands': 'NA',
    # CAF
    'Algeria': 'AF', 'Angola': 'AF', 'Benin': 'AF', 'Botswana': 'AF', 'Burkina Faso': 'AF',
    'Burundi': 'AF', 'Cameroon': 'AF', 'Cape Verde': 'AF', 'Central African Republic': 'AF',
    'Chad': 'AF', 'Comoros': 'AF', 'Congo': 'AF', 'DR Congo': 'AF', 'Djibouti': 'AF',
    'Egypt': 'AF', 'Equatorial Guinea': 'AF', 'Eritrea': 'AF', 'Eswatini': 'AF',
    'Ethiopia': 'AF', 'Gabon': 'AF', 'Gambia': 'AF', 'Ghana': 'AF', 'Guinea': 'AF',
    'Guinea-Bissau': 'AF', 'Ivory Coast': 'AF', 'Kenya': 'AF', 'Lesotho': 'AF',
    'Liberia': 'AF', 'Libya': 'AF', 'Madagascar': 'AF', 'Malawi': 'AF', 'Mali': 'AF',
    'Mauritania': 'AF', 'Mauritius': 'AF', 'Morocco': 'AF', 'Mozambique': 'AF',
    'Namibia': 'AF', 'Niger': 'AF', 'Nigeria': 'AF', 'Rwanda': 'AF',
    'São Tomé and Príncipe': 'AF', 'Senegal': 'AF', 'Seychelles': 'AF', 'Sierra Leone': 'AF',
    'Somalia': 'AF', 'South Africa': 'AF', 'South Sudan': 'AF', 'Sudan': 'AF',
    'Tanzania': 'AF', 'Togo': 'AF', 'Tunisia': 'AF', 'Uganda': 'AF', 'Zambia': 'AF',
    'Zimbabwe': 'AF',
    # AFC (note: Australia moved from OFC to AFC in 2006)
    'Afghanistan': 'AS', 'Australia': 'AS', 'Bahrain': 'AS', 'Bangladesh': 'AS', 'Bhutan': 'AS',
    'Brunei': 'AS', 'Cambodia': 'AS', 'China PR': 'AS', 'East Timor': 'AS', 'Guam': 'AS',
    'Hong Kong': 'AS', 'India': 'AS', 'Indonesia': 'AS', 'Iran': 'AS', 'Iraq': 'AS',
    'Japan': 'AS', 'Jordan': 'AS', 'Kuwait': 'AS', 'Kyrgyzstan': 'AS', 'Laos': 'AS',
    'Lebanon': 'AS', 'Macau': 'AS', 'Malaysia': 'AS', 'Maldives': 'AS', 'Mongolia': 'AS',
    'Myanmar': 'AS', 'Nepal': 'AS', 'North Korea': 'AS', 'Oman': 'AS', 'Pakistan': 'AS',
    'Palestine': 'AS', 'Philippines': 'AS', 'Qatar': 'AS', 'Saudi Arabia': 'AS',
    'Singapore': 'AS', 'South Korea': 'AS', 'Sri Lanka': 'AS', 'Syria': 'AS',
    'Tajikistan': 'AS', 'Thailand': 'AS', 'Turkmenistan': 'AS', 'United Arab Emirates': 'AS',
    'Uzbekistan': 'AS', 'Vietnam': 'AS', 'Yemen': 'AS',
    # OFC
    'American Samoa': 'OC', 'Cook Islands': 'OC', 'Fiji': 'OC', 'New Caledonia': 'OC',
    'New Zealand': 'OC', 'Papua New Guinea': 'OC', 'Samoa': 'OC', 'Solomon Islands': 'OC',
    'Tahiti': 'OC', 'Tonga': 'OC', 'Vanuatu': 'OC',
}

host_continent = df['country'].map(CONTINENT)
home_continent = df['home_team'].map(CONTINENT)
away_continent = df['away_team'].map(CONTINENT)

df['home_in_host_continent'] = (home_continent == host_continent).astype(int)
df['away_in_host_continent'] = (away_continent == host_continent).astype(int)

# drop the temp country col — same choice we made earlier
df = df.drop(columns=['country'])

df[['date', 'home_team', 'away_team', 'neutral',
    'home_in_host_continent', 'away_in_host_continent']].tail(10)


,date,home_team,away_team,neutral,home_in_host_continent,away_in_host_continent
5735,2026-06-26,Cape Verde,Saudi Arabia,True,0,0
5736,2026-06-26,Uruguay,Spain,True,0,0
5737,2026-06-26,Norway,France,True,0,0
5738,2026-06-26,Senegal,Iraq,True,0,0
5739,2026-06-27,Algeria,Austria,True,0,0
5740,2026-06-27,Jordan,Argentina,True,0,0
5741,2026-06-27,Colombia,Portugal,True,0,0
5742,2026-06-27,DR Congo,Uzbekistan,True,0,0
5743,2026-06-27,Panama,England,True,1,0
5744,2026-06-27,Croatia,Ghana,True,0,0


### Add sofifa data

In [54]:
import re
# Sofifa uses a few non-standard names. Map them to whatever results.csv uses
# so this stats table can later be joined to df by team name.
SOFIFA_TO_CANONICAL = {
    'Czechia': 'Czech Republic',
    'Türkiye': 'Turkey',
    'Cabo Verde': 'Cape Verde',
    'Congo DR': 'DR Congo',
    "Côte d'Ivoire": 'Ivory Coast',
    'Korea Republic': 'South Korea',
    'Curacao': 'Curaçao',
}

_ROW_PATTERN = re.compile(
    r'<a href="https://sofifa\.com/team/\d+/[^/]+/\d+/">([^<]+)</a>'
    r'.*?data-col="oa".*?title="(\d+)"'
    r'.*?data-col="at".*?title="(\d+)"'
    r'.*?data-col="md".*?title="(\d+)"'
    r'.*?data-col="df".*?title="(\d+)"'
    r'.*?data-col="sa"><em>([\d.]+)</em>',
    re.DOTALL,
)

def parse_sofifa_listing(filepath: str) -> pd.DataFrame:
    """Pull team / overall / attack / midfield / defence / starting-XI age
    from a saved sofifa national-teams listing page."""
    with open(filepath, encoding='utf-8') as f:
        html = f.read()
    rows = _ROW_PATTERN.findall(html)
    out = pd.DataFrame(rows, columns=[
        'team', 'overall', 'attack', 'midfield', 'defence', 'starting_xi_avg_age'
    ])
    out['team'] = out['team'].replace(SOFIFA_TO_CANONICAL)
    for c in ['overall', 'attack', 'midfield', 'defence']:
        out[c] = pd.to_numeric(out[c])
    out['starting_xi_avg_age'] = pd.to_numeric(out['starting_xi_avg_age'])
    return out

fc25 = parse_sofifa_listing('sofifa_teams_fc25.html')
fc26 = parse_sofifa_listing('sofifa_teams_fc26.html')
print(f"FC25 parsed: {len(fc25)} teams; FC26 parsed: {len(fc26)} teams")
fc26.head()


FC25 parsed: 29 teams; FC26 parsed: 60 teams


,team,overall,attack,midfield,defence,starting_xi_avg_age
0,France,86,87,85,85,25.82
1,Spain,85,85,86,83,25.45
2,England,84,86,85,83,25.27
3,Germany,84,81,84,85,26.18
4,Portugal,84,84,86,84,27.36


In [55]:
# 2015-2024 snapshots from Stefano's CSV. league_id 78 = "Friendly International"
# (i.e. the national-team rows). FIFA version N → calendar year 2000 + N.

CSV_TO_CANONICAL = {
    'Korea Republic': 'South Korea',
}

teams_hist = pd.read_csv(
    'male_teams_14-24.csv',
    usecols=['team_name', 'fifa_version', 'league_id',
             'overall', 'attack', 'midfield', 'defence',
             'starting_xi_average_age'],
)
teams_hist = teams_hist[teams_hist['league_id'] == 78].drop(columns=['league_id'])
teams_hist['team'] = teams_hist['team_name'].replace(CSV_TO_CANONICAL)
teams_hist['year'] = 2000 + teams_hist['fifa_version'].astype(int)
teams_hist = teams_hist.rename(columns={'starting_xi_average_age': 'starting_xi_avg_age'})
teams_hist = (teams_hist[['team', 'year', 'overall', 'attack', 'midfield', 'defence',
                          'starting_xi_avg_age']]
              .drop_duplicates(subset=['team', 'year'])
              .reset_index(drop=True))
teams_hist.tail()


,team,year,overall,attack,midfield,defence,starting_xi_avg_age
431,Egypt,2015,69,70,68,67,25.45
432,Bulgaria,2015,67,66,67,66,28.18
433,Bolivia,2015,65,68,60,63,27.73
434,New Zealand,2015,64,65,59,63,24.36
435,India,2015,57,59,56,56,25.82


In [56]:
# Combine into one (team, year) table. FC25 had ~30 teams missing from the scrape —
# back-fill those with their FC24 row (best available proxy) so the 2025 slice is
# complete. FC26 is taken as-is.

fc24 = teams_hist.loc[teams_hist['year'] == 2024,
                      ['team', 'overall', 'attack', 'midfield', 'defence',
                       'starting_xi_avg_age']]

fc25_filled = pd.concat([
    fc25.assign(year=2025),
    fc24[~fc24['team'].isin(fc25['team'])].assign(year=2025),
], ignore_index=True)

fc26_yr = fc26.assign(year=2026)

team_stats = pd.concat([teams_hist, fc25_filled, fc26_yr], ignore_index=True)
team_stats = (team_stats[['team', 'year', 'overall', 'attack', 'midfield',
                          'defence', 'starting_xi_avg_age']]
              .sort_values(['team', 'year'])
              .reset_index(drop=True))

print(f"rows: {len(team_stats)} | teams: {team_stats['team'].nunique()} "
      f"| years: {team_stats['year'].min()}-{team_stats['year'].max()}")
team_stats[team_stats['team'].isin(['France', 'Brazil', 'Argentina', 'Cape Verde'])]


rows: 526 | teams: 72 | years: 2015-2026


,team,year,overall,attack,midfield,defence,starting_xi_avg_age
1,Argentina,2015,81,84,80,79,27.73
2,Argentina,2016,83,88,81,82,27.64
3,Argentina,2017,84,87,84,80,28.55
4,Argentina,2018,83,88,82,81,29.27
5,Argentina,2019,83,87,82,81,28.82
6,Argentina,2020,81,84,81,79,27.55
7,Argentina,2021,82,87,80,79,28.09
8,Argentina,2022,84,87,82,81,27.82
9,Argentina,2023,83,86,82,81,28.73
10,Argentina,2024,83,85,83,83,28.64


In [57]:
# Attach team_stats to each match using actual sofifa snapshot dates so a match
# played in (e.g.) Aug 2024 picks the FIFA 24 row, not the FC 25 row that hadn't
# released yet. 2015-2024 dates come from the CSV's update_as_of column;
# FC 25 / FC 26 use their public release dates.

SNAPSHOT_DATE = {
    2015: '2014-09-18', 2016: '2015-09-21', 2017: '2016-09-20',
    2018: '2017-09-18', 2019: '2018-08-21', 2020: '2019-09-19',
    2021: '2020-09-23', 2022: '2021-09-23', 2023: '2022-09-26',
    2024: '2023-09-22', 2025: '2024-09-27', 2026: '2025-09-26',
}

ts = team_stats.copy()
ts['snapshot_date'] = pd.to_datetime(ts['year'].map(SNAPSHOT_DATE))
ts = ts.drop(columns=['year']).sort_values('snapshot_date').reset_index(drop=True)

home_stats = ts.rename(columns={
    'team': 'home_team',
    'overall': 'home_overall', 'attack': 'home_attack',
    'midfield': 'home_midfield', 'defence': 'home_defence',
    'starting_xi_avg_age': 'home_starting_xi_avg_age',
})
away_stats = ts.rename(columns={
    'team': 'away_team',
    'overall': 'away_overall', 'attack': 'away_attack',
    'midfield': 'away_midfield', 'defence': 'away_defence',
    'starting_xi_avg_age': 'away_starting_xi_avg_age',
})

df_with_stats = pd.merge_asof(
    df.sort_values('date').reset_index(drop=True),
    home_stats,
    left_on='date', right_on='snapshot_date',
    by='home_team',
    direction='backward',
).drop(columns=['snapshot_date'])

df_with_stats = pd.merge_asof(
    df_with_stats.sort_values('date').reset_index(drop=True),
    away_stats,
    left_on='date', right_on='snapshot_date',
    by='away_team',
    direction='backward',
).drop(columns=['snapshot_date'])

n = len(df_with_stats)
print(f"NaN home_overall: {df_with_stats['home_overall'].isna().sum()}/{n} "
      f"| NaN away_overall: {df_with_stats['away_overall'].isna().sum()}/{n}")
df_with_stats[['date', 'home_team', 'away_team',
               'home_overall', 'home_attack', 'home_defence',
               'away_overall', 'away_attack', 'away_defence']].tail(10)


NaN home_overall: 3739/5745 | NaN away_overall: 3984/5745


,date,home_team,away_team,home_overall,home_attack,home_defence,away_overall,away_attack,away_defence
5735,2026-06-26,Cape Verde,Saudi Arabia,70.0,69.0,71.0,72.0,74.0,71.0
5736,2026-06-26,Uruguay,Spain,79.0,78.0,79.0,85.0,85.0,83.0
5737,2026-06-26,Norway,France,79.0,80.0,75.0,86.0,87.0,85.0
5738,2026-06-26,Senegal,Iraq,79.0,79.0,77.0,65.0,63.0,64.0
5739,2026-06-27,Algeria,Austria,76.0,75.0,76.0,78.0,77.0,79.0
5740,2026-06-27,Panama,England,70.0,70.0,69.0,84.0,86.0,83.0
5741,2026-06-27,Jordan,Argentina,66.0,66.0,66.0,83.0,84.0,79.0
5742,2026-06-27,DR Congo,Uzbekistan,74.0,78.0,75.0,68.0,70.0,71.0
5743,2026-06-27,Colombia,Portugal,78.0,79.0,78.0,84.0,84.0,84.0
5744,2026-06-27,Croatia,Ghana,79.0,79.0,78.0,75.0,78.0,73.0


## Visualizations